In [1]:
import duckdb

con = duckdb.connect("../data/fieldview.duckdb", read_only=True)

con.sql("SHOW TABLES")

┌─────────────────────────┐
│          name           │
│         varchar         │
├─────────────────────────┤
│ statsapi_people         │
│ statsapi_roster         │
│ statsapi_stats_hitting  │
│ statsapi_stats_pitching │
│ statsapi_teams          │
└─────────────────────────┘

In [2]:
# -- position variety: is it a clean 9-way split, or is there an "OF"/"IF"/"UTIL" 
# -- bucket that'll complicate the OF-subs-OF, IF-subs-IF grouping you asked for?
query = """
SELECT position_abbreviation, COUNT(*) 
FROM statsapi_roster GROUP BY 1 ORDER BY 2 DESC;
"""

con.sql(query).df()

,position_abbreviation,count_star()
0,P,390
1,C,63
2,CF,48
3,2B,48
4,RF,45
5,LF,44
6,SS,42
7,3B,41
8,1B,38
9,DH,21


In [3]:
# -- roster <-> stats match rate: how many of the 780 roster players actually 
# -- have a stats row at all (rookies/just-called-up guys legitimately won't)
query = """
SELECT COUNT(*) AS roster_total,
       SUM(CASE WHEN EXISTS (SELECT 1 FROM statsapi_stats_hitting h WHERE h.person_id = r.person_id)
                 OR EXISTS (SELECT 1 FROM statsapi_stats_pitching p WHERE p.person_id = r.person_id)
           THEN 1 ELSE 0 END) AS matched
FROM statsapi_roster r;
"""

con.sql(query).df()

,roster_total,matched
0,782,779.0
